In [1]:
import lightgbm as lgb
import pandas as pd
import optuna
import warnings
import json

from i import input_dir, model_dir
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

train = pd.read_csv(input_dir + "train.csv", index_col="id")
test = pd.read_csv(input_dir + "test.csv", index_col="id")
origin = pd.read_csv(input_dir + "diabetes_dataset.csv")

for col in train.select_dtypes(include="object").columns:
    train[col] = train[col].astype("category")
    test[col] = test[col].astype("category")
for col in origin.select_dtypes(include="object").columns:
    origin[col] = origin[col].astype("category")

Target_Col = "diagnosed_diabetes"

X = train.iloc[:, :-1]
y = train[Target_Col]
X_test = test

X_orig = origin[X.columns]
y_orig = origin[Target_Col]

with open(model_dir + "lgbm_base_params.json") as f:
    base_params = json.load(f)

c:\Users\Blanc\AppData\Local\pypoetry\Cache\virtualenvs\playground-lwZmZsxv-py3.12\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Make sure features are float32 for GPU speed
# X = X.astype("float32")
# X_test = X_test.astype("float32")
# X_orig = X_orig.astype("float32")

# Convert categorical columns explicitly (LightGBM uses pandas category dtype)
cat_cols = X.select_dtypes(["category"]).columns.tolist()


def objective(trial):
    boosting_type = trial.suggest_categorical("boosting_type", ["gbdt", "dart"])

    param = {
        "objective": "binary",
        "metric": "auc",
        "verbosity": -1,
        "boosting_type": boosting_type,
        # GPU acceleration
        "device": "gpu",
        "gpu_platform_id": 0,
        "gpu_device_id": 0,
        # Learning
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 3000),
        # Tree structure
        "num_leaves": trial.suggest_int("num_leaves", 16, 512),
        "max_depth": trial.suggest_int("max_depth", -1, 16),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 200),
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 10.0, log=True),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 5.0),
        # Regularization
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        # Sampling
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 0, 10),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        # Histogram / tree
        "max_bin": trial.suggest_int("max_bin", 32, 255),
        "grow_policy": trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"]),
        "extra_trees": trial.suggest_categorical("extra_trees", [False, True]),
        # Dart-specific
        "drop_rate": trial.suggest_float("drop_rate", 0.0, 0.5) if boosting_type == "dart" else 0.0,
        "skip_drop": trial.suggest_float("skip_drop", 0.0, 0.5) if boosting_type == "dart" else 0.0,
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs = []

    for train_idx, valid_idx in cv.split(X, y):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        model = lgb.LGBMClassifier(**param, verbose=-1)

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            eval_metric="auc",
            categorical_feature=cat_cols,  # Pass categorical columns
            callbacks=[lgb.early_stopping(50), optuna.integration.LightGBMPruningCallback(trial, "auc")],  # <-- pruning
        )

        preds = model.predict_proba(X_valid)[:, 1]
        aucs.append(roc_auc_score(y_valid, preds))

    return sum(aucs) / len(aucs)


# Run the tuner
study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner())
study.enqueue_trial(base_params)
study.optimize(objective, n_trials=100)

print("Best params:", study.best_params)
print("Best AUC:", study.best_value)


[I 2025-12-01 19:08:18,524] A new study created in memory with name: no-name-8298e957-d10a-482f-92a1-f3b584cc0c7f


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[604]	valid_0's auc: 0.727282
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[866]	valid_0's auc: 0.726561
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[899]	valid_0's auc: 0.727574
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[601]	valid_0's auc: 0.728135
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[758]	valid_0's auc: 0.727578


[I 2025-12-01 19:12:11,646] Trial 0 finished with value: 0.7274257631309758 and parameters: {'boosting_type': 'gbdt', 'learning_rate': 0.0339749309455783, 'n_estimators': 1666, 'num_leaves': 320, 'max_depth': 15, 'min_data_in_leaf': 149, 'min_child_weight': 0.04064536464251674, 'min_split_gain': 2.123164590414576, 'lambda_l1': 4.1217653404278493e-05, 'lambda_l2': 6.792285700816295e-06, 'bagging_fraction': 0.6001912137374683, 'bagging_freq': 8, 'feature_fraction': 0.7993832169466938, 'max_bin': 255, 'grow_policy': 'depthwise', 'extra_trees': False}. Best is trial 0 with value: 0.7274257631309758.


Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1075]	valid_0's auc: 0.701315
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1075]	valid_0's auc: 0.699515
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1075]	valid_0's auc: 0.699374
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1070]	valid_0's auc: 0.700817
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1072]	valid_0's auc: 0.701198


[I 2025-12-01 19:12:54,091] Trial 1 finished with value: 0.7004439522987107 and parameters: {'boosting_type': 'gbdt', 'learning_rate': 0.03862984584130251, 'n_estimators': 1075, 'num_leaves': 404, 'max_depth': 1, 'min_data_in_leaf': 109, 'min_child_weight': 0.010242705690347659, 'min_split_gain': 4.97718781281073, 'lambda_l1': 2.0173246171875428e-07, 'lambda_l2': 0.00015604659617935707, 'bagging_fraction': 0.7911775805810879, 'bagging_freq': 0, 'feature_fraction': 0.5856231991778387, 'max_bin': 32, 'grow_policy': 'lossguide', 'extra_trees': False}. Best is trial 0 with value: 0.7274257631309758.


Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2101]	valid_0's auc: 0.707367
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1499]	valid_0's auc: 0.704111
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2106]	valid_0's auc: 0.70506
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1898]	valid_0's auc: 0.706404
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1503]	valid_0's auc: 0.706521


[I 2025-12-01 19:14:36,443] Trial 2 finished with value: 0.7058886862518091 and parameters: {'boosting_type': 'gbdt', 'learning_rate': 0.11746073257219981, 'n_estimators': 2106, 'num_leaves': 42, 'max_depth': 10, 'min_data_in_leaf': 80, 'min_child_weight': 0.025025379479419137, 'min_split_gain': 4.828331142830874, 'lambda_l1': 0.13307668629612882, 'lambda_l2': 6.006798171072726e-06, 'bagging_fraction': 0.7230912250039253, 'bagging_freq': 8, 'feature_fraction': 0.6469580785770945, 'max_bin': 140, 'grow_policy': 'depthwise', 'extra_trees': True}. Best is trial 0 with value: 0.7274257631309758.


In [ ]:
best_params = study.best_params
with open(model_dir + "lgbm_base_params.json", "w") as f:
    json.dump(best_params, f, indent=4)